# 06. Cross-National Comparison — Reproducible Final Version

This notebook combines forecast evaluation, SARIMAX intervention estimates, residual diagnostics, and counterfactual gaps into a cross-country evidence table.

The purpose is not to claim that one model universally dominates. The purpose is to compare how anxiety-related search behavior differs across the six Anglophone countries.

## A. Import and Path

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path('modeling_outputs')
FIGURES_DIR = Path('figures')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 160)

COUNTRY_DISPLAY = {
    'Australia': 'Australia',
    'Canada': 'Canada',
    'Ireland': 'Ireland',
    'New_Zealand': 'New Zealand',
    'New Zealand': 'New Zealand',
    'UK': 'United Kingdom',
    'US': 'United States',
}

MODEL_DISPLAY = {
    'Naive': 'Naive',
    'Seasonal_Naive': 'Seasonal Naive',
    'Seasonal Naive': 'Seasonal Naive',
    'ARIMA': 'ARIMA',
    'SARIMA': 'SARIMA',
    'SARIMAX': 'SARIMAX',
}

HORIZON_DISPLAY = {
    'h01_03': 'h01-03',
    'h04_12': 'h04-12',
    'h13_24': 'h13-24',
    'h25_plus': 'h25+',
}

def read_result(name: str) -> pd.DataFrame:
    candidates = [RESULTS_DIR / name, Path(name)]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path)
    searched = ' or '.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Cannot find {name}. Searched: {searched}. Run Notebook 03 first, then run this notebook.')

def require_columns(df: pd.DataFrame, cols, table_name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}. Available columns: {list(df.columns)}')

def add_display_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'country' in out.columns:
        out['country_label'] = out['country'].map(COUNTRY_DISPLAY).fillna(out['country'])
    if 'model_type' in out.columns:
        out['model_label'] = out['model_type'].map(MODEL_DISPLAY).fillna(out['model_type'])
    if 'best_test_model' in out.columns:
        out['best_test_model_label'] = out['best_test_model'].map(MODEL_DISPLAY).fillna(out['best_test_model'])
    if 'horizon_group' in out.columns:
        out['horizon_label'] = out['horizon_group'].map(HORIZON_DISPLAY).fillna(out['horizon_group'])
    return out


## B. Load final component tables

In [2]:

best = read_result('04_best_test_forecast_model_by_country.csv')
model_eval = read_result('03_final_anxiety_model_class_comparison_enhanced.csv')
sig = read_result('05_significant_sarimax_intervention_terms.csv')
diag = read_result('05_full_sample_sarimax_residual_diagnostics.csv')
logit_phase = read_result('05_logit_counterfactual_by_country_phase.csv')
horizon = read_result('04_horizon_specific_average_errors.csv')
rolling = read_result('04_rolling_origin_average_performance.csv')

# Standardize the best-model table regardless of whether 04 kept original column names.
if 'best_test_model' not in best.columns:
    if 'model_type' in best.columns:
        best = best.rename(columns={'model_type': 'best_test_model'})
    else:
        raise ValueError('04_best_test_forecast_model_by_country.csv must contain model_type or best_test_model.')
if 'best_test_RMSE' not in best.columns and 'test_RMSE' in best.columns:
    best = best.rename(columns={'test_RMSE': 'best_test_RMSE'})
if 'best_test_sMAPE' not in best.columns and 'test_sMAPE' in best.columns:
    best = best.rename(columns={'test_sMAPE': 'best_test_sMAPE'})

require_columns(best, ['country', 'best_test_model', 'best_test_RMSE', 'best_test_sMAPE'], '04_best_test_forecast_model_by_country.csv')
require_columns(sig, ['country', 'term', 'coef', 'pvalue', 'direction'], '05_significant_sarimax_intervention_terms.csv')
require_columns(diag, ['country', 'LjungBox_p24', 'ARCH_LM_p12', 'JarqueBera_p', 'lb24_pass_5pct'], '05_full_sample_sarimax_residual_diagnostics.csv')
require_columns(logit_phase, ['country', 'counterfactual_version', 'phase', 'mean_gap', 'percent_gap_vs_counterfactual_mean', 'share_months_above_counterfactual', 'share_months_above_upper_95'], '05_logit_counterfactual_by_country_phase.csv')
require_columns(horizon, ['horizon_group', 'model_type', 'RMSE'], '04_horizon_specific_average_errors.csv')
require_columns(rolling, ['model_type', 'mean_RMSE'], '04_rolling_origin_average_performance.csv')

display(best)
display(diag)


,country,best_test_model,order,seasonal_order,exog_set,AIC_refit,BIC_refit,test_MAE,best_test_RMSE,best_test_sMAPE,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,status,country_label,model_label
0,Australia,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,3.903226,5.220926,4.512262,221,25.877828,24.840372,0.0,0.0,1.150542e-34,0.000005,OK,Australia,Seasonal Naive
1,Canada,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,4.225806,5.064105,4.941757,221,25.122172,24.549921,0.0,0.0,6.219140e-36,0.000005,OK,Canada,Seasonal Naive
2,Ireland,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,4.096774,5.509523,5.252198,221,26.407240,26.976932,0.0,0.0,1.191312e-34,0.000009,OK,Ireland,Seasonal Naive
3,New_Zealand,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,6.387097,8.226629,7.401847,221,26.678733,26.641747,0.0,0.0,4.474915e-34,0.000069,OK,New Zealand,Seasonal Naive
4,UK,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,3.741935,4.711003,4.422434,221,30.153846,28.847937,0.0,0.0,2.227417e-36,0.000004,OK,United Kingdom,Seasonal Naive
5,US,Naive,NaN,NaN,NaN,NaN,NaN,3.193548,4.299287,3.510255,232,-42.051724,23.185062,0.0,0.0,7.145778e-40,0.000004,OK,United States,Naive


,country,model_scope,model_type,order,seasonal_order,exog_set,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,lb24_pass_5pct,arch_pass_5pct,jb_pass_5pct,country_label,model_label
0,Australia,full_sample_sarimax_intervention,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.102630,4.183652,0.062631,1.005659e-01,6.270711e-13,1.807964e-160,True,False,False,Australia,SARIMAX
1,Canada,full_sample_sarimax_intervention,SARIMAX,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.053696,4.376930,0.019799,2.976631e-09,1.323972e-02,0.000000e+00,False,False,False,Canada,SARIMAX
2,Ireland,full_sample_sarimax_intervention,SARIMAX,"(2, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.116920,4.694010,0.980827,8.510330e-01,1.176280e-05,1.408227e-82,True,False,False,Ireland,SARIMAX
3,New_Zealand,full_sample_sarimax_intervention,SARIMAX,"(0, 1, 1)","(1, 0, 0, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.071871,5.678170,0.038927,1.466380e-01,5.837526e-01,1.564583e-11,True,True,False,New Zealand,SARIMAX
4,UK,full_sample_sarimax_intervention,SARIMAX,"(2, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.037586,3.698107,0.200473,4.389158e-07,4.101939e-05,3.144295e-23,False,False,False,United Kingdom,SARIMAX
5,US,full_sample_sarimax_intervention,SARIMAX,"(1, 1, 1)","(1, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.083259,3.440927,0.063361,1.434593e-01,3.667517e-20,0.000000e+00,True,False,False,United States,SARIMAX


## C. Build country scorecard

In [3]:

countries = sorted(best['country'].dropna().unique())

sig_rows = []
for country in countries:
    sub = sig.loc[sig['country'].eq(country)].copy()
    if sub.empty:
        txt = 'No significant SARIMAX intervention terms at 5%.'
    else:
        txt = '; '.join([f"{r.term}: {r.direction} (coef={r.coef:.3f}, p={r.pvalue:.3f})" for r in sub.itertuples()])
    sig_rows.append({'country': country, 'significant_sarimax_terms_5pct': txt})
sig_df = pd.DataFrame(sig_rows)

logit_full = (
    logit_phase.loc[(logit_phase['counterfactual_version'].eq('logit')) & (logit_phase['phase'].eq('Full_2020_2025')),
                    ['country', 'mean_gap', 'percent_gap_vs_counterfactual_mean', 'share_months_above_counterfactual', 'share_months_above_upper_95']]
    .rename(columns={
        'mean_gap': 'logit_full_mean_gap',
        'percent_gap_vs_counterfactual_mean': 'logit_full_percent_gap',
        'share_months_above_counterfactual': 'logit_share_above_counterfactual',
        'share_months_above_upper_95': 'logit_share_above_upper_95',
    })
)

rmse_pivot = model_eval.pivot_table(index='country', columns='model_type', values='test_RMSE', aggfunc='first').reset_index()
rmse_pivot.columns = [str(c) for c in rmse_pivot.columns]

score = best.merge(sig_df, on='country', how='left')
score = score.merge(diag[['country', 'LjungBox_p24', 'ARCH_LM_p12', 'JarqueBera_p', 'lb24_pass_5pct', 'arch_pass_5pct', 'jb_pass_5pct']], on='country', how='left')
score = score.merge(logit_full, on='country', how='left')
score = score.merge(rmse_pivot, on='country', how='left')
score = add_display_columns(score)
score.to_csv(RESULTS_DIR / '06_crossnational_country_scorecard.csv', index=False)
display(score)


,country,best_test_model,order,seasonal_order,exog_set,AIC_refit,BIC_refit,test_MAE,best_test_RMSE,best_test_sMAPE,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24_x,ARCH_LM_p12_x,JarqueBera_p_x,status,country_label,model_label,significant_sarimax_terms_5pct,LjungBox_p24_y,ARCH_LM_p12_y,JarqueBera_p_y,lb24_pass_5pct,arch_pass_5pct,jb_pass_5pct,logit_full_mean_gap,logit_full_percent_gap,logit_share_above_counterfactual,logit_share_above_upper_95,ARIMA,Naive,SARIMA,SARIMAX,Seasonal_Naive,best_test_model_label
0,Australia,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,3.903226,5.220926,4.512262,221,25.877828,24.840372,0.0,0.0,1.150542e-34,0.000005,OK,Australia,Seasonal Naive,"covid_trend: negative (coef=-0.519, p=0.030); ...",1.005659e-01,6.270711e-13,1.807964e-160,True,False,False,-7.982221,-8.532420,0.097222,0.027778,12.874957,15.892786,8.630130,38.805024,5.220926,Seasonal Naive
1,Canada,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,4.225806,5.064105,4.941757,221,25.122172,24.549921,0.0,0.0,6.219140e-36,0.000005,OK,Canada,Seasonal Naive,"post_covid_trend: negative (coef=-0.647, p=0.002)",2.976631e-09,1.323972e-02,0.000000e+00,False,False,False,-2.948809,-3.277308,0.347222,0.055556,8.992276,5.877788,12.302622,21.635925,5.064105,Seasonal Naive
2,Ireland,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,4.096774,5.509523,5.252198,221,26.407240,26.976932,0.0,0.0,1.191312e-34,0.000009,OK,Ireland,Seasonal Naive,No significant SARIMAX intervention terms at 5%.,8.510330e-01,1.176280e-05,1.408227e-82,True,False,False,-14.474367,-15.314192,0.013889,0.000000,9.989977,6.463296,9.206978,32.844506,5.509523,Seasonal Naive
3,New_Zealand,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,6.387097,8.226629,7.401847,221,26.678733,26.641747,0.0,0.0,4.474915e-34,0.000069,OK,New Zealand,Seasonal Naive,No significant SARIMAX intervention terms at 5%.,1.466380e-01,5.837526e-01,1.564583e-11,True,True,False,-8.031212,-8.458548,0.180556,0.041667,15.025975,17.199962,17.197161,9.728385,8.226629,Seasonal Naive
4,UK,Seasonal_Naive,NaN,NaN,NaN,NaN,NaN,3.741935,4.711003,4.422434,221,30.153846,28.847937,0.0,0.0,2.227417e-36,0.000004,OK,United Kingdom,Seasonal Naive,"early_covid_shock: negative (coef=-9.869, p=0....",4.389158e-07,4.101939e-05,3.144295e-23,False,False,False,-3.655185,-3.995845,0.319444,0.097222,10.284168,6.140821,11.822028,19.445630,4.711003,Seasonal Naive
5,US,Naive,NaN,NaN,NaN,NaN,NaN,3.193548,4.299287,3.510255,232,-42.051724,23.185062,0.0,0.0,7.145778e-40,0.000004,OK,United States,Naive,"post_covid_trend: negative (coef=-0.497, p=0.049)",1.434593e-01,3.667517e-20,0.000000e+00,True,False,False,0.714394,0.823831,0.555556,0.152778,6.963280,4.299287,9.213932,16.556629,7.202150,Naive


## D. Add interpretation labels

In [4]:

def forecast_label(row):
    if row['best_test_model'] in ['Naive', 'Seasonal_Naive', 'Seasonal Naive']:
        return 'benchmark-dominant forecast case'
    return f"{MODEL_DISPLAY.get(row['best_test_model'], row['best_test_model'])}-dominant forecast case"

def intervention_label(txt):
    if isinstance(txt, str) and txt.startswith('No significant'):
        return 'no significant SARIMAX intervention terms'
    if isinstance(txt, str) and 'negative' in txt and 'positive' not in txt:
        return 'negative or declining intervention evidence'
    if isinstance(txt, str) and 'positive' in txt and 'negative' not in txt:
        return 'positive intervention evidence'
    return 'mixed intervention evidence'

def diagnostics_label(row):
    if pd.isna(row.get('LjungBox_p24')):
        return 'residual diagnostics unavailable'
    return 'residuals pass Ljung-Box p24' if row['LjungBox_p24'] > 0.05 else 'residual autocorrelation remains'

def counterfactual_label(row):
    if pd.isna(row.get('logit_full_mean_gap')):
        return 'counterfactual unavailable'
    return 'observed above logit counterfactual on average' if row['logit_full_mean_gap'] > 0 else 'observed below logit counterfactual on average'

labeled = score.copy()
labeled['forecast_label'] = labeled.apply(forecast_label, axis=1)
labeled['intervention_label'] = labeled['significant_sarimax_terms_5pct'].apply(intervention_label)
labeled['diagnostics_label'] = labeled.apply(diagnostics_label, axis=1)
labeled['counterfactual_label'] = labeled.apply(counterfactual_label, axis=1)
labeled.to_csv(RESULTS_DIR / '06_crossnational_country_scorecard_labeled.csv', index=False)
display(labeled[['country', 'country_label', 'best_test_model', 'best_test_RMSE', 'intervention_label', 'diagnostics_label', 'counterfactual_label']])


,country,country_label,best_test_model,best_test_RMSE,intervention_label,diagnostics_label,counterfactual_label
0,Australia,Australia,Seasonal_Naive,5.220926,negative or declining intervention evidence,residual diagnostics unavailable,observed below logit counterfactual on average
1,Canada,Canada,Seasonal_Naive,5.064105,negative or declining intervention evidence,residual diagnostics unavailable,observed below logit counterfactual on average
2,Ireland,Ireland,Seasonal_Naive,5.509523,no significant SARIMAX intervention terms,residual diagnostics unavailable,observed below logit counterfactual on average
3,New_Zealand,New Zealand,Seasonal_Naive,8.226629,no significant SARIMAX intervention terms,residual diagnostics unavailable,observed below logit counterfactual on average
4,UK,United Kingdom,Seasonal_Naive,4.711003,negative or declining intervention evidence,residual diagnostics unavailable,observed below logit counterfactual on average
5,US,United States,Naive,4.299287,negative or declining intervention evidence,residual diagnostics unavailable,observed above logit counterfactual on average


## E. Report-ready cross-national summary

In [5]:

bench_count = int(best['best_test_model'].isin(['Naive', 'Seasonal_Naive', 'Seasonal Naive']).sum())
seasonal_count = int(best['best_test_model'].isin(['Seasonal_Naive', 'Seasonal Naive']).sum())

h01 = horizon.loc[horizon['horizon_group'].eq('h01_03')].sort_values('RMSE')
sarima_short = h01.iloc[0]['model_type'] if not h01.empty else 'not available'
roll_best = rolling.sort_values('mean_RMSE').iloc[0]['model_type'] if not rolling.empty else 'not available'

n_sig_terms = int(len(sig))
n_sig_countries = int(sig['country'].nunique()) if not sig.empty else 0
negative_sig = int((sig['direction'] == 'negative').sum()) if not sig.empty else 0
positive_sig = int((sig['direction'] == 'positive').sum()) if not sig.empty else 0
pass_count = int(diag['lb24_pass_5pct'].sum())
neg_logit = int((logit_full['logit_full_mean_gap'] < 0).sum()) if not logit_full.empty else 0

summary = pd.DataFrame([
    {
        'topic': 'Forecasting',
        'finding': f'Benchmark models are best in {bench_count}/6 post-COVID test countries; Seasonal Naive is best in {seasonal_count}/6.',
        'interpretation': 'Post-COVID anxiety search behavior is highly persistent and seasonally regular; forecast evaluation validates the modeling context rather than replacing intervention analysis.',
    },
    {
        'topic': 'ARIMA/SARIMA modeling',
        'finding': f'{MODEL_DISPLAY.get(sarima_short, sarima_short)} is the best average model for horizons h01-03; {MODEL_DISPLAY.get(roll_best, roll_best)} has the best rolling-origin average RMSE.',
        'interpretation': 'SARIMA remains useful for short-horizon dynamics and seasonal structure even when benchmarks dominate long-horizon test forecasts.',
    },
    {
        'topic': 'SARIMAX intervention',
        'finding': f'{n_sig_terms} significant intervention terms appear across {n_sig_countries}/6 countries ({negative_sig} negative; {positive_sig} positive).',
        'interpretation': 'The SARIMAX evidence does not support a robust uniform positive COVID-period increase; significant results mainly indicate decline or normalization.',
    },
    {
        'topic': 'Counterfactual analysis',
        'finding': f'The logit counterfactual shows negative full-period mean gaps in {neg_logit}/6 countries.',
        'interpretation': 'Observed anxiety searches usually fall below the no-COVID baseline after accounting for the bounded Google Trends scale.',
    },
    {
        'topic': 'Diagnostics',
        'finding': f'Full-sample SARIMAX residuals pass Ljung-Box p24 in {pass_count}/6 countries.',
        'interpretation': 'Inference is stronger in countries that pass residual autocorrelation diagnostics; countries that fail require cautious interpretation.',
    },
])
summary.to_csv(RESULTS_DIR / '06_crossnational_summary_for_report.csv', index=False)

# Compact final findings table for direct use in report writing.
final_findings = labeled[['country', 'country_label', 'best_test_model', 'best_test_RMSE', 'forecast_label', 'intervention_label', 'diagnostics_label', 'counterfactual_label']].copy()
final_findings.to_csv(RESULTS_DIR / '06_final_findings_for_report.csv', index=False)

display(summary)
display(final_findings)


,topic,finding,interpretation
0,Forecasting,Benchmark models are best in 6/6 post-COVID te...,Post-COVID anxiety search behavior is highly p...
1,ARIMA/SARIMA modeling,SARIMA is the best average model for horizons ...,SARIMA remains useful for short-horizon dynami...
2,SARIMAX intervention,7 significant intervention terms appear across...,The SARIMAX evidence does not support a robust...
3,Counterfactual analysis,The logit counterfactual shows negative full-p...,Observed anxiety searches usually fall below t...
4,Diagnostics,Full-sample SARIMAX residuals pass Ljung-Box p...,Inference is stronger in countries that pass r...


,country,country_label,best_test_model,best_test_RMSE,forecast_label,intervention_label,diagnostics_label,counterfactual_label
0,Australia,Australia,Seasonal_Naive,5.220926,benchmark-dominant forecast case,negative or declining intervention evidence,residual diagnostics unavailable,observed below logit counterfactual on average
1,Canada,Canada,Seasonal_Naive,5.064105,benchmark-dominant forecast case,negative or declining intervention evidence,residual diagnostics unavailable,observed below logit counterfactual on average
2,Ireland,Ireland,Seasonal_Naive,5.509523,benchmark-dominant forecast case,no significant SARIMAX intervention terms,residual diagnostics unavailable,observed below logit counterfactual on average
3,New_Zealand,New Zealand,Seasonal_Naive,8.226629,benchmark-dominant forecast case,no significant SARIMAX intervention terms,residual diagnostics unavailable,observed below logit counterfactual on average
4,UK,United Kingdom,Seasonal_Naive,4.711003,benchmark-dominant forecast case,negative or declining intervention evidence,residual diagnostics unavailable,observed below logit counterfactual on average
5,US,United States,Naive,4.299287,benchmark-dominant forecast case,negative or declining intervention evidence,residual diagnostics unavailable,observed above logit counterfactual on average
